# Omni NCM — 82.9M, both arms, one stream, one pass

Both arms (`strict` level-parallel, `ar` flat) train in **one process off one live Hub stream**.
No produce step, no corpus: the batch is built once and stepped by each arm, so identical rows,
order and packing are structural rather than bookkeeping. Held-out blocks come off the same
stream at the start.

**Measured, not assumed:** 40 steps, both arms, checkpoints written, ~1.5 samples/s of stream in
one process. That rate is what sets the sample budget below, not the GPU.



In [ ]:
import os, sys, subprocess

REPO = 'adi274903/ncm-omni'          # private
PKG  = os.environ.get('NCM_PKG', '/content/ncm_omni')

def _token():
    """Read-only clone access. Colab secret first, then env, then an authenticated gh."""
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t:
            return t.strip()
    except Exception:
        pass
    t = os.environ.get('GITHUB_TOKEN')
    if t:
        return t.strip()
    try:
        return subprocess.check_output(['gh', 'auth', 'token'], text=True).strip()
    except Exception:
        return None

def _askpass(tok):
    """Hand the token to git through GIT_ASKPASS, never through argv or .git/config.

    The old cell pulled with the plain https URL, but the clone deliberately stripped the
    credential out of origin - so a re-run died with `could not read Username`. The token has to
    be re-supplied on every pull, and it must not land in the notebook output or on disk.
    """
    d = os.path.dirname(os.path.abspath(PKG))
    secret = os.path.join(d, '.git-tok')
    with open(secret, 'w') as f:
        f.write(tok)
    os.chmod(secret, 0o600)
    p = os.path.join(d, '.git-askpass.sh')
    with open(p, 'w') as f:
        f.write('#!/bin/sh\ncase "$1" in\n  *sername*) echo x-access-token ;;\n'
                '  *) cat "{}" ;;\nesac\n'.format(secret))
    os.chmod(p, 0o700)
    return p, secret

def _git(args, tok=None):
    env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
    junk = []
    if tok:
        ap, secret = _askpass(tok)
        junk = [ap, secret]
        env.update(GIT_ASKPASS=ap, GIT_ASKPASS_REQUIRE='force')
    try:
        # credential.helper= clears any stale helper so askpass is the only source
        return subprocess.check_output(['git', '-c', 'credential.helper='] + args,
                                       text=True, stderr=subprocess.STDOUT, env=env).strip()
    finally:
        for f in junk:
            if os.path.exists(f):
                os.remove(f)

assert PKG.startswith('/content') or os.environ.get('NCM_PKG'), 'refusing to touch ' + PKG
tok = _token()
url = 'https://github.com/%s.git' % REPO
if os.path.isdir(os.path.join(PKG, '.git')):
    try:
        # Fetch + hard reset, not pull: the checkout is a --depth 1 graft, so a rewound history
        # cannot fast-forward, and run/cfg is rewritten by every invocation. The repo holds no
        # edits worth keeping (run/ is gitignored), so landing exactly on origin/main is right.
        _git(['-C', PKG, 'fetch', '--depth', '1', 'origin', 'main'], tok)
        _git(['-C', PKG, 'reset', '--hard', 'FETCH_HEAD'], tok)
    except subprocess.CalledProcessError as e:
        # Re-clone, but keep run/ (shards + checkpoints). A plain `rm -rf` here silently deletes
        # hours of training because run/ is gitignored and therefore not in the repo.
        print('update failed -> re-cloning (keeping run/):',
              str(e.output).strip().splitlines()[-1])
        import shutil
        tmp = PKG.rstrip('/') + '.new'
        shutil.rmtree(tmp, ignore_errors=True)
        _git(['clone', '--depth', '1', url, tmp], tok)
        keep = os.path.join(PKG, 'run')
        if os.path.isdir(keep):
            shutil.rmtree(os.path.join(tmp, 'run'), ignore_errors=True)
            shutil.move(keep, os.path.join(tmp, 'run'))
        shutil.rmtree(PKG, ignore_errors=True)
        os.rename(tmp, PKG)
else:
    _git(['clone', '--depth', '1', url, PKG], tok)

assert os.path.exists(os.path.join(PKG, 'core.py')), 'clone landed without core.py'
os.chdir(PKG)
if os.environ.get('NCM_SKIP_PIP') != '1':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                           'datasets', 'soundfile', 'av'])

cfg = os.path.join(PKG, '.git', 'config')
assert 'x-access-token' not in open(cfg).read(), 'credential leaked into .git/config'

import torch
print('repo   ', REPO, '| commit', subprocess.check_output(
    ['git', '-C', PKG, 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('torch  ', torch.__version__, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('workdir', os.getcwd())


In [ ]:
# HF token + hardware probe. Unauthenticated Hub requests answer 429 on bursts of stream opens,
# which silently drops sources and changes the mix. The token is read from Colab secrets and
# never printed.
import os, shutil, subprocess, sys

def _hf_token():
    try:
        from google.colab import userdata
        for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            try:
                t = userdata.get(k)
                if t:
                    return t.strip()
            except Exception:
                pass
    except Exception:
        pass
    for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k):
            return os.environ[k].strip()
    return None

t = _hf_token()
if t:
    os.environ["HF_TOKEN"] = t           # picked up by huggingface_hub in-process and by children
    print("HF token: loaded (%d chars)" % len(t))
else:
    print("HF token: MISSING -> add a Colab secret named HF_TOKEN "
          "(key icon, left sidebar). Without it the producer will 429 and skip sources.")

try:
    from huggingface_hub import whoami
    print("HF auth: ", whoami()["name"] if os.environ.get("HF_TOKEN") else "anonymous")
except Exception as e:
    print("HF auth check skipped:", type(e).__name__)

cpu = os.cpu_count()
ram = shutil.disk_usage("/").total / 1e9
try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
except Exception:
    gpu, vram = "unknown", 0
def _ram_gb():
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemTotal"):
                return int(line.split()[1]) / 1e6
    except Exception:
        pass
    return 0.0
print(f"cpu {cpu} threads | RAM {_ram_gb():.0f} GB | disk {ram:.0f} GB | {gpu} ({vram} GB)")
print("WORKERS note: 8 measured at 45 samples/s on a 12-vCPU runtime with a token and a clean process table.")


In [ ]:
# Invariants at the configured schedule. Must print PIPELINE CHECK OK.
!python run_omni.py check 2>&1 | tail -8

In [ ]:
# ---- CONFIG: the only cell you edit ------------------------------------------------
PRESET    = "md"        # md = 82.88M (d768 L13 h12 kv3). "sm" = 10.44M debug, "lg" = 454M
BASE_LEN  = 64          # 504 packed positions/sample. 1024 => 7,119 pos and a memory wall
BATCH     = 16          # both arms step this batch. Drop to 8 on a T4
HOLDOUT   = 4           # x4 => 128 text / 96 image / 64 audio / 32 video eval rows
SAMPLE_BUDGET = 30000   # samples, seen exactly once by both arms
SAVE_EVERY = 200        # live stream has no replay: checkpoints are the only restart

PARAMS_MD, POS_PER_SAMPLE = 82.88e6, 504
STEPS = SAMPLE_BUDGET // BATCH
print(f"{PRESET} | {BASE_LEN} -> {POS_PER_SAMPLE} pos/sample | {STEPS:,} steps x {BATCH} = "
      f"{STEPS*BATCH:,} samples, seen exactly once by both arms")
print(f"{STEPS*BATCH*POS_PER_SAMPLE/1e6:.1f}M positions per arm "
      f"({STEPS*BATCH*POS_PER_SAMPLE/PARAMS_MD:.2f} per parameter)")
print("measured stream rate in one process: ~1.5 samples/s on a 16-thread box.")
print("=> this run is ~{:.0f} h if Colab streams at the same rate; watch pos_per_s and".format(
      SAMPLE_BUDGET/1.5/3600))
print("   extrapolate after step 100. Full 1.0 pos/param (167k samples) needs a shard corpus:")
print("   62 min of produce buys 45 samples/s instead of 1.5, i.e. 31 h -> 1 h.")


## Train both arms off one stream

There is no produce step and no corpus. Both arms are stepped from the same batch inside one
process, which is a stricter matched-params control than two runs over a shared shard set: the
rows, their order and the packing are identical by construction, not by careful bookkeeping.

The cost is no replay. A crash means streaming again from the start, which is why `save_every`
is set in the config and why a kill test — short and judged on one comparison — is the right
place for this. A long pretraining run would want shards back.

First hundred steps look bad on purpose: the stream opens 20 Hub datasets and the encoders warm
up, so early `pos_per_s` is meaningless.


In [ ]:
# ---- TRAIN: both arms, one stream, one pass -----------------------------------------
# No shard corpus. The batch is built once and stepped by every arm in this process, so
# "same data, same order, same packing" is true by construction rather than by discipline.
# The held-out blocks are carved off the same stream at the start (worker 0 of the stream),
# so eval cannot drift from training either.
import os, subprocess, sys, time

BPASS = f"--preset {PRESET} --base-len {BASE_LEN}"
cmd = (f"{sys.executable} -u run_omni.py train --arms strict,ar {BPASS} "
       f"--batch {BATCH} --steps {STEPS} --live --holdout-mult {HOLDOUT} --save-every {SAVE_EVERY}")
print(cmd)
t0 = time.time()
PKG = os.environ.get("NCM_PKG", "/content/ncm_omni")
r = subprocess.run(cmd, shell=True, cwd=PKG)      # repo root, not the notebook cwd
print(f"exit {r.returncode} after {(time.time()-t0)/60:.1f} min")
for arm in ("strict", "ar"):
    f = os.path.join(PKG, f"run/train_{arm}/final.pt")
    print(arm, "checkpoint:", os.path.exists(f))


In [ ]:
# ---- EVAL: 320 held-out rows, conditional table only -----------------------------
!python run_omni.py eval --preset md --base-len 64 --holdout-mult 4 2>&1 | tee run/eval.log | tail -45


In [ ]:
# ---- VERDICT: the level-parallel margin, on real data ------------------------------
import json, os

tab = json.load(open("run/eval_table.json"))
cond = tab["conditional"]
st = cond.get("level-parallel(strict)", {})
ar = cond.get("autoregressive", {})
floor = tab["baseline_by_level"].get("5", tab["baseline"])

print(f"{'level':>6} | {'strict':>7} | {'AR':>7} | {'margin':>7}")
for k in range(6):
    key = f"text_L{k}"
    if key in st:
        a = ar.get(key, float("nan"))
        print(f"{k:>6} | {st[key]:7.3f} | {a:7.3f} | {st[key] - a:+7.3f}")
print(f"\nconstant floor, text L5: {floor:.3f}")
for m in ("image", "audio", "video"):
    if f"{m}_L5" in st:
        print(f"{m:>7} L5: strict {st[f'{m}_L5']:.3f} | AR {ar.get(f'{m}_L5', float('nan')):.3f}")

margin = st.get("text_L5", float("nan")) - ar.get("text_L5", float("nan"))
print(f"\nmargin at text L5: {margin:+.3f}")
print()
if margin != margin:
    print("VERDICT: no margin - an arm is missing from run/train_*. Re-run the missing train cell.")
elif st.get("text_L5", 0) <= floor:
    print("VERDICT: KILL. strict does not beat the constant floor. The level-parallel claim fails "
          "at 82.9M on real data.")
elif margin < 0.05:
    print("VERDICT: KILL. No usable advantage over a matched-params AR baseline at this scale.")
elif margin < 0.10:
    print("VERDICT: WEAK. A real but small margin. Report it, do not spend on 315M yet - raise "
          "positions per parameter first and see if it widens.")
else:
    print("VERDICT: HOLDS. strict beats AR by >=0.10 at 82.9M on 167k unique samples. That is the "
          "justification for the 315M run, which needs chunked attention first.")

for d in ("/content/drive/MyDrive/ncm", "/content", "."):
    try:
        os.makedirs(d, exist_ok=True)
        dst = os.path.join(d, "eval_table_killtest.json")
        json.dump(tab, open(dst, "w"), indent=2)
        print("\nsaved", dst)
        break
    except Exception:
        continue


## Reading the result

- **The margin, not the absolute number.** Absolute accuracy depends on the eval set, which just
  changed size (32 -> 128 text rows), so 0.519 is no longer directly comparable. The margin is.
- **Per modality, never the mean.** Video was dead at 10M (0.129 vs 0.183 floor) and audio barely
  cleared its floor; if they are still below the constant, say so rather than averaging them in.
- **Free-running stays void.** Level 0 is a 1-of-256 guess; eval labels any margin under 0.05 a tie.
- **This is the first clean run.** Anything quoted from the 4k/11k stream predates the
  `ShardStream` fix and trained on a single shard; do not put those numbers in a paper.
- **1 position per parameter is not Chinchilla.** Both arms are under-trained by design — the run
  is long enough to compare architectures, not to produce a good model. Say that in the paper.

## If something breaks

| symptom | cause | fix |
|---|---|---|
| `CUDA out of memory` in train | batch too large for the card | `--batch 8`, or `BATCH = 8` and re-run cell 4's step count |
| produce stops growing, no error | Hub throttling, or a sharded read is stuck | cell 4 self-stops after 7 min; check `tail run/produce.log` |
| eval prints `VOID` everywhere | level-0 free-running is a coin flip | expected; only the conditional table carries the claim |
| a `LEGACY-leaky` row appears in eval | an old `train_block` checkpoint survived | `rm -rf run/train_block` and re-run eval |
